#  GeoVision-CLIP Cali — Entrenamiento Situación 2

**Notebook autocontenido para Lightning AI (GPU gratuita)**

Todas las definiciones del modelo están inline. No requiere `src/models/`.
Solo necesita acceso a internet y credenciales GCP.

---

## 1. Instalación de dependencias

>  Ejecutar UNA sola vez al iniciar la sesión en Lightning AI.

In [ ]:
!pip install -q open-clip-torch transformers sentence-transformers gcsfs zarr pandas matplotlib tqdm scikit-learn
import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
PyTorch 2.10.0+cpu | CUDA: False


## 2. Autenticación Google Cloud Storage

El dataset está en `gs://geovision-cali/datasets/sit2/`. Necesitas credenciales GCP.

### Opción A: Service Account JSON (recomendado)
Sube tu JSON de service account a Lightning AI y ejecuta:

In [20]:
import os
import gcsfs

SERVICE_ACCOUNT_JSON = 'proyecto3ia-494900-fa5d413358ff.json'

if os.path.exists(SERVICE_ACCOUNT_JSON):
    try:
        fs = gcsfs.GCSFileSystem(project='proyecto3ia-494900', token=SERVICE_ACCOUNT_JSON)

        # Ajustamos a la carpeta que SÍ existe en el bucket
        path_situacion2 = 'geovision-cali/sit2_dataset/'
        files = fs.ls(path_situacion2)

        print(f'GCS OK: {len(files)} archivos encontrados en sit2_dataset')
        for f in files:
            print(f'  - {f}')

    except Exception as e:
        print(f"Error al listar la carpeta: {e}")
else:
    print('Error: No se encontró el archivo JSON.')

GCS OK: 6 archivos encontrados en sit2_dataset
  - geovision-cali/sit2_dataset/RESUMEN.json
  - geovision-cali/sit2_dataset/pares_metadata.parquet
  - geovision-cali/sit2_dataset/percentiles_s5p.json
  - geovision-cali/sit2_dataset/secuencias_8fechas.parquet
  - geovision-cali/sit2_dataset/splits
  - geovision-cali/sit2_dataset/tiles.zarr


## 3. Imports y configuración base

In [21]:
import json, hashlib, time, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import gcsfs, zarr
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from transformers import AutoModel, AutoTokenizer
import open_clip
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

Dispositivo: cpu


## 4. Constantes del proyecto

Parámetros según la consigna: α=0.1, λ=1e-3, τ=0.07, d_sae=512, proj_dim=256

In [22]:
PROJECT_GCP = 'proyecto3ia-494900'
BUCKET = 'geovision-cali'
DATASET_PREFIX = 'datasets/sit2'

CLASES = ['contaminacion_alta_NO2','contaminacion_alta_SO2','ozono_anomalo','vegetacion_densa','suelo_urbano']

VISUAL_DIM = 512
TEXT_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
TEXT_DIM = 384
SAE_DIM = 512
PROJ_DIM = 256
LAMBDA_L1 = 1e-3
ALPHA = 0.1
TEMPERATURE = 0.07

BATCH_SIZE = 16
LR = 1e-4
WEIGHT_DECAY = 1e-6
EPOCHS = 60
WARMUP_EPOCHS = 5
GRADIENT_ACCUMULATION_STEPS = 2
EARLY_STOPPING_PATIENCE = 15

OUTPUT_DIR = Path('checkpoints')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Configuracion cargada OK')

Configuracion cargada OK


## 5. Sparse Autoencoder (SAE)

**Paper:** Muhamed et al. (2025) — NAACL | **Paper:** Zaigrajew et al. (2025) — ICML

$$L_{sae} = \|x - \hat{x}\|^2 + \lambda \cdot \|z\|_1$$

- Encoder: $z = \text{ReLU}(W_{enc} x + b_{enc})$
- Decoder: $\hat{x} = W_{dec} z$ (sin bias, columnas normalizadas a norma 1)

In [23]:
"""Sparse Autoencoder para interpretabilidad de embeddings.

Implementa el SAE según Muhamed et al. (2025) — NAACL 2025 y
Zaigrajew et al. (2025) — ICML 2025 (Matryoshka SAE para CLIP).

L_sae = ‖x − x̂‖² + λ · ‖z‖₁

Referencias:
- Muhamed et al. (2025): "Decoding Dark Matter: Specialized Sparse
  Autoencoders for Interpreting Rare Concepts in Foundation Models"
- Zaigrajew et al. (2025): "Interpreting CLIP with Hierarchical
  Sparse Autoencoders" (ICML 2025, arXiv:2502.20578)
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class SparseAutoencoder(nn.Module):
    """Autoencoder esparso con regularización L1 y normalización del decoder.

    Args:
        d_model: Dimensión del espacio de entrada/salida (512 para ViT-B/32).
        d_sae:   Dimensión latente del SAE (default: igual a d_model).
        lambda_l1: Peso de regularización L1 (consigna: 1e-3).

    Uso:
        sae = SparseAutoencoder(d_model=512, lambda_l1=1e-3)
        z, x_hat = sae(x)          # forward
        loss, l_rec, l_sparse = sae.loss(x, z, x_hat)
        sae.normalize_decoder()    # después de cada step del optimizer
    """

    def __init__(
        self,
        d_model: int = 512,
        d_sae: int = 512,
        lambda_l1: float = 1e-3,
    ) -> None:
        super().__init__()
        self.W_enc = nn.Linear(d_model, d_sae, bias=True)
        self.W_dec = nn.Linear(d_sae, d_model, bias=False)
        self.lambda_l1 = lambda_l1

        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.xavier_uniform_(self.W_enc.weight)
        nn.init.zeros_(self.W_enc.bias)
        nn.init.orthogonal_(self.W_dec.weight)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Forward pass del SAE.

        Args:
            x: Tensor de entrada (..., d_model).

        Returns:
            z: Representación latente esparsa (..., d_sae).
            x_hat: Reconstrucción (..., d_model).
        """
        z = F.relu(self.W_enc(x))      # encoder: sparse activation
        x_hat = self.W_dec(z)           # decoder: reconstruct
        return z, x_hat

    def loss(
        self, x: torch.Tensor, z: torch.Tensor, x_hat: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Calcula la pérdida del SAE.

        L = ‖x − x̂‖² + λ · ‖z‖₁

        Returns:
            total: Pérdida total.
            l_rec: Pérdida de reconstrucción (MSE).
            l_sparse: Pérdida de esparsidad (L1).
        """
        l_rec = F.mse_loss(x_hat, x)
        l_sparse = self.lambda_l1 * z.abs().mean()
        return l_rec + l_sparse, l_rec, l_sparse

    @torch.no_grad()
    def normalize_decoder(self) -> None:
        """Normaliza columnas de W_dec a norma unitaria.

        Previene colapso de features (Bricken et al., 2023;
        Zaigrajew et al., 2025). Debe llamarse tras cada
        optimizer.step().
        """
        norms = self.W_dec.weight.norm(dim=0, keepdim=True)
        self.W_dec.weight.div_(norms.clamp(min=1e-8))

    def get_sparsity_ratio(self, z: torch.Tensor, threshold: float = 0.01) -> float:
        """Fracción de features inactivas: mean(|z| < threshold).

        KPI consigna: ≥ 0.70 mínimo, ≥ 0.85 excelente.
        """
        return (z.abs() < threshold).float().mean().item()

    def get_l0(self, z: torch.Tensor) -> float:
        """Número medio de features activas por muestra.

        Métrica complementaria del paper MSAE (Zaigrajew et al., 2025).
        """
        return (z.abs() > 0.01).float().sum(dim=-1).mean().item()


## 6. BandAdapter + GeoVisionCLIP

**Paper:** SenCLIP (Jain et al., WACV 2025) — BandAdapter
| **Paper:** RemoteCLIP (Liu et al., 2024 · IEEE TGRS) — CLIP architecture

In [24]:
"""Modelo GeoVision-CLIP: arquitectura multimodal para teledetección.

Combina:
- ViT-B/32 (RemoteCLIP) como encoder visual
- paraphrase-multilingual-MiniLM como encoder textual
- Sparse Autoencoders simétricos (visual + textual)
- Projection heads para espacio contrastivo ℝ²⁵⁶

Referencias:
- Liu et al. (2024): RemoteCLIP (IEEE TGRS)
- Jain et al. (2025): SenCLIP (WACV 2025, arXiv:2412.08536)
- Muhamed et al. (2025): SSAE (NAACL 2025)
"""


import torch
import torch.nn as nn
import torch.nn.functional as F



# ---------------------------------------------------------------------------
# Band Adapter: 12 bandas Sentinel-2 → 3 canales RGB-equivalentes
# ---------------------------------------------------------------------------
class BandAdapter(nn.Module):
    """Proyecta 12 bandas S2 a 3 canales RGB-equivalentes.

    Inspirado en SenCLIP (Jain et al., WACV 2025). Inicializado para
    aproximar true-color (B4→R, B3→G, B2→B) y refinado durante
    entrenamiento contrastivo.

    Bandas S2 (GEE orden): B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12
    (se excluye SCL, índice 12 en el orden completo de 13 bandas).
    """

    def __init__(self, in_bands: int = 12, out_channels: int = 3):
        super().__init__()
        self.conv = nn.Conv2d(in_bands, out_channels, kernel_size=1, bias=True)
        self._init_true_color()

    def _init_true_color(self) -> None:
        """Inicializa para true-color: B4→R(0), B3→G(1), B2→B(2)."""
        with torch.no_grad():
            self.conv.weight.zero_()
            self.conv.bias.zero_()
            self.conv.weight[0, 3] = 1.0   # B4 → Red
            self.conv.weight[1, 2] = 1.0   # B3 → Green
            self.conv.weight[2, 1] = 1.0   # B2 → Blue
        # Las demás bandas (RE, NIR, SWIR) quedan libres para aprender

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 12, H, W) → (B, 3, H, W)"""
        return self.conv(x)


# ---------------------------------------------------------------------------
# GeoVision-CLIP: modelo completo
# ---------------------------------------------------------------------------
class GeoVisionCLIP(nn.Module):
    """Modelo multimodal CLIP + SAE para teledetección.

    Arquitectura según consigna del proyecto:
    - Encoder visual: ViT-B/32 (RemoteCLIP frozen) + BandAdapter
    - Encoder textual: paraphrase-multilingual-MiniLM-L12-v2
    - SAE visual y textual (d=512, λ=1e-3)
    - Projection heads: 512 → 256 con L2 norm
    """

    def __init__(
        self,
        visual_encoder: nn.Module,
        text_encoder: nn.Module,
        text_tokenizer,
        visual_dim: int = 512,
        text_dim: int = 512,          # después de proyección interna
        sae_dim: int = 512,
        proj_dim: int = 256,
        lambda_l1: float = 1e-3,
        freeze_visual: bool = True,
    ) -> None:
        super().__init__()

        self.visual_encoder = visual_encoder
        self.text_encoder = text_encoder
        self.tokenizer = text_tokenizer

        # SAEs simétricos
        self.sae_img = SparseAutoencoder(d_model=visual_dim, d_sae=sae_dim, lambda_l1=lambda_l1)
        self.sae_txt = SparseAutoencoder(d_model=text_dim, d_sae=sae_dim, lambda_l1=lambda_l1)

        # Projection heads: ℝᵈ → ℝ²⁵⁶
        self.proj_img = nn.Sequential(
            nn.Linear(sae_dim, proj_dim),
            nn.LayerNorm(proj_dim),
        )
        self.proj_txt = nn.Sequential(
            nn.Linear(sae_dim, proj_dim),
            nn.LayerNorm(proj_dim),
        )

        if freeze_visual:
            for p in self.visual_encoder.parameters():
                p.requires_grad = False

    def encode_image(self, tiles: torch.Tensor) -> torch.Tensor:
        """tiles: (B, 12, 64, 64) → e_img: (B, 256) L2-normalized"""
        h = self.visual_encoder(tiles)          # (B, 512)
        z, _ = self.sae_img(h)                   # (B, 512)
        e = self.proj_img(z)                     # (B, 256)
        return F.normalize(e, dim=-1)

    def encode_text(self, texts: list[str]) -> torch.Tensor:
        """texts: list[str] → e_txt: (B, 256) L2-normalized"""
        tokens = self.tokenizer(
            texts, padding=True, truncation=True, max_length=77, return_tensors="pt"
        )
        device = next(self.text_encoder.parameters()).device
        tokens = {k: v.to(device) for k, v in tokens.items()}
        out = self.text_encoder(**tokens)
        h = out.last_hidden_state.mean(dim=1)    # mean pooling → (B, text_dim)
        z, _ = self.sae_txt(h)                    # (B, 512)
        e = self.proj_txt(z)                      # (B, 256)
        return F.normalize(e, dim=-1)

    def forward(
        self, tiles: torch.Tensor, texts: list[str]
    ) -> dict[str, torch.Tensor]:
        """Forward completo con todos los outputs intermedios.

        Returns dict con:
            e_img, e_txt: embeddings contrastivos (B, 256)
            h_img, h_txt: embeddings pre-SAE (B, 512)
            z_img, z_txt: representaciones latentes SAE (B, 512)
            x_hat_img, x_hat_txt: reconstrucciones SAE (B, 512)
        """
        # Rama visual
        h_img = self.visual_encoder(tiles)        # (B, 512)
        z_img, x_hat_img = self.sae_img(h_img)    # (B, 512)
        e_img = F.normalize(self.proj_img(z_img), dim=-1)

        # Rama textual
        tokens = self.tokenizer(
            texts, padding=True, truncation=True, max_length=77, return_tensors="pt"
        )
        device = next(self.text_encoder.parameters()).device
        tokens = {k: v.to(device) for k, v in tokens.items()}
        out = self.text_encoder(**tokens)
        h_txt = out.last_hidden_state.mean(dim=1)  # (B, text_dim)
        z_txt, x_hat_txt = self.sae_txt(h_txt)     # (B, 512)
        e_txt = F.normalize(self.proj_txt(z_txt), dim=-1)

        return {
            "e_img": e_img,
            "e_txt": e_txt,
            "h_img": h_img,
            "h_txt": h_txt,
            "z_img": z_img,
            "z_txt": z_txt,
            "x_hat_img": x_hat_img,
            "x_hat_txt": x_hat_txt,
        }

    def normalize_decoders(self) -> None:
        """Normaliza columnas de W_dec en ambos SAEs."""
        self.sae_img.normalize_decoder()
        self.sae_txt.normalize_decoder()


## 7. Función de Pérdida

$$L_{total} = L_{InfoNCE} + \alpha \cdot (L_{sae}^{img} + L_{sae}^{txt})$$

- $\alpha = 0.1$, $\tau = 0.07$ (learnable)
- InfoNCE bidireccional (img→txt + txt→img) / 2

In [25]:
"""Funciones de pérdida para GeoVision-CLIP.

L_total = L_InfoNCE + α · (L_sae_img + L_sae_txt)

Referencias:
- Radford et al. (2021): CLIP (InfoNCE)
- Liu et al. (2024): RemoteCLIP (IEEE TGRS) — InfoNCE para satélite
"""


import torch
import torch.nn as nn
import torch.nn.functional as F


class GeoVisionCLIPLoss(nn.Module):
    """Pérdida total del modelo GeoVision-CLIP.

    Combina pérdida contrastiva InfoNCE con pérdidas de reconstrucción
    y esparsidad de los Sparse Autoencoders.

    Args:
        temperature: Temperatura inicial para InfoNCE (learnable, init 0.07).
        alpha: Peso de las pérdidas SAE (consigna: 0.1).
    """

    def __init__(self, temperature: float = 0.07, alpha: float = 0.1) -> None:
        super().__init__()
        self.logit_scale = nn.Parameter(torch.tensor(1.0 / temperature).log())
        self.alpha = alpha

    def info_nce(
        self, img_emb: torch.Tensor, txt_emb: torch.Tensor
    ) -> torch.Tensor:
        """InfoNCE bidireccional imagen↔texto.

        Args:
            img_emb: Embeddings visuales (B, d) ya normalizados.
            txt_emb: Embeddings textuales (B, d) ya normalizados.

        Returns:
            Pérdida InfoNCE promedio (imagen→texto + texto→imagen) / 2.
        """
        logit_scale = self.logit_scale.exp()
        logits = logit_scale * (img_emb @ txt_emb.T)
        labels = torch.arange(len(img_emb), device=img_emb.device)
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        return (loss_i2t + loss_t2i) / 2

    def forward(
        self,
        e_img: torch.Tensor,
        e_txt: torch.Tensor,
        sae_img_loss: torch.Tensor,
        sae_txt_loss: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Calcula la pérdida total.

        Args:
            e_img, e_txt: Embeddings contrastivos normalizados.
            sae_img_loss: Pérdida total del SAE visual (reconstrucción + L1).
            sae_txt_loss: Pérdida total del SAE textual.

        Returns:
            total: Pérdida total L_total.
            l_clip: Componente contrastiva.
            l_sae_img, l_sae_txt: Componentes SAE.
        """
        l_clip = self.info_nce(e_img, e_txt)
        l_sae_img = sae_img_loss
        l_sae_txt = sae_txt_loss
        total = l_clip + self.alpha * (l_sae_img + l_sae_txt)
        return total, l_clip, l_sae_img, l_sae_txt


## 8. Métricas (KPIs de la consigna)

| KPI | Mínimo | Excelente |
|-----|--------|-----------|
| Recall@1 | ≥ 0.45 | ≥ 0.65 |
| Recall@5 | ≥ 0.70 | ≥ 0.85 |
| Sparsity | ≥ 0.70 | ≥ 0.85 |
| MSE rec | ≤ 0.05 | ≤ 0.02 |

In [26]:
"""Métricas de evaluación para GeoVision-CLIP.

KPIs de la consigna (Situación 2):
- Recall@1 (img→txt): ≥ 0.45 mín, ≥ 0.65 exc
- Recall@5 (img→txt): ≥ 0.70 mín, ≥ 0.85 exc
- Sparsity ratio SAE:   ≥ 0.70 mín, ≥ 0.85 exc
- MSE reconstrucción:     ≤ 0.05 mín, ≤ 0.02 exc

Referencias:
- Liu et al. (2024): RemoteCLIP — métricas de retrieval (Table II)
- Muhamed et al. (2025): SSAE — sparsity metrics
- Zaigrajew et al. (2025): MSAE — FVU, cosine similarity
"""


import torch
import torch.nn.functional as F


def recall_at_k(
    sim_matrix: torch.Tensor, k: int = 1
) -> float:
    """Calcula Recall@k para retrieval imagen→texto.

    Args:
        sim_matrix: Matriz de similitud (N_query, N_gallery).
        k: Número de resultados a considerar.

    Returns:
        Fracción de queries donde el ground truth está en el top-k.
    """
    n = sim_matrix.size(0)
    labels = torch.arange(n, device=sim_matrix.device)
    topk = sim_matrix.topk(k, dim=1).indices
    correct = (topk == labels.unsqueeze(1)).any(dim=1).float()
    return correct.mean().item()


def sparsity_ratio(z: torch.Tensor, threshold: float = 0.01) -> float:
    """Fracción de features inactivas: mean(|z| < threshold).

    KPI consigna: ≥ 0.70 mínimo, ≥ 0.85 excelente.
    """
    return (z.abs() < threshold).float().mean().item()


def cosine_sim_reconstruction(x: torch.Tensor, x_hat: torch.Tensor) -> float:
    """Similitud coseno media entre original y reconstrucción.

    Métrica complementaria de Zaigrajew et al. (2025) — MSAE.
    Valores > 0.95 indican reconstrucción de alta calidad.
    """
    return F.cosine_similarity(x, x_hat, dim=-1).mean().item()


def fraction_variance_unexplained(x: torch.Tensor, x_hat: torch.Tensor) -> float:
    """FVU = Var(x - x̂) / Var(x).

    Métrica de Zaigrajew et al. (2025). < 0.1 excelente.
    """
    var_res = (x - x_hat).var(dim=0).sum()
    var_x = x.var(dim=0).sum()
    return (var_res / (var_x + 1e-8)).item()


def compute_all_kpis(
    e_img: torch.Tensor,
    e_txt: torch.Tensor,
    z_img: torch.Tensor,
    z_txt: torch.Tensor,
    h_img: torch.Tensor,
    h_txt: torch.Tensor,
    x_hat_img: torch.Tensor,
    x_hat_txt: torch.Tensor,
) -> dict[str, float]:
    """Calcula todos los KPIs de la Situación 2.

    Args:
        e_img, e_txt: Embeddings contrastivos normalizados (B, 256).
        z_img, z_txt: Representaciones SAE (B, 512).
        h_img, h_txt: Embeddings pre-SAE originales (B, 512).
        x_hat_img, x_hat_txt: Reconstrucciones SAE (B, 512).

    Returns:
        Dict con todos los KPIs.
    """
    sim = e_img @ e_txt.T

    return {
        "recall@1": recall_at_k(sim, k=1),
        "recall@5": recall_at_k(sim, k=5),
        "sparsity_img": sparsity_ratio(z_img),
        "sparsity_txt": sparsity_ratio(z_txt),
        "mse_rec_img": F.mse_loss(x_hat_img, h_img).item(),
        "mse_rec_txt": F.mse_loss(x_hat_txt, h_txt).item(),
        "cos_sim_rec_img": cosine_sim_reconstruction(h_img, x_hat_img),
        "cos_sim_rec_txt": cosine_sim_reconstruction(h_txt, x_hat_txt),
        "fvu_img": fraction_variance_unexplained(h_img, x_hat_img),
        "fvu_txt": fraction_variance_unexplained(h_txt, x_hat_txt),
    }


## 2. Dataset y DataLoader

In [28]:
class GeoVisionDataset(Dataset):
    """Dataset de pares imagen-texto para GeoVision-CLIP.

    Carga bajo demanda desde GCS:
    - tiles desde tiles.zarr (N, 13, 64, 64) int16
    - metadatos desde metadatos.parquet
    """

    def __init__(self, split: str):
        self.split = split

        # Cargar metadata
        fs = gcsfs.GCSFileSystem(project=PROJECT_GCP)
        self.df = pd.read_parquet(f"gs://{BUCKET}/{DATASET_PREFIX}/metadatos.parquet")
        self.df = self.df[self.df["split"] == split].reset_index(drop=True)

        # Abrir tiles.zarr (acceso lazy, solo carga lo necesario)
        mapper = fs.get_mapper(f"{BUCKET}/{DATASET_PREFIX}/tiles.zarr")
        self.tiles = zarr.open(mapper, mode="r")

        # Mapa clase → índice
        self.clase_a_idx = {c: i for i, c in enumerate(CLASES)}

        # Estadísticas para normalización (precalculadas o estimadas)
        self._compute_norm_stats()

        print(f" Dataset {split}: {len(self)} pares")
        print(f"   Balance: {self.df['clase'].value_counts().to_dict()}")

    def _compute_norm_stats(self):
        """Estima min/max por banda para normalización [0, 1]."""
        # Muestrear 200 tiles aleatorios para estimar estadísticas
        n_sample = min(200, len(self))
        idxs = np.random.choice(len(self), n_sample, replace=False)
        samples = []
        for i in tqdm(idxs, desc="Estimando estadísticas de normalización"):
            tile = self.tiles[i]  # (13, 64, 64) int16
            samples.append(tile[:12].astype("float32"))  # sin SCL

        all_data = np.concatenate([s.reshape(12, -1) for s in samples], axis=1)  # (12, N*pix)
        self.band_mins = all_data.min(axis=1)   # (12,)
        self.band_maxs = all_data.max(axis=1)   # (12,)
        # Evitar división por cero
        self.band_maxs = np.maximum(self.band_maxs, self.band_mins + 1)

        print(f"   Mins: {self.band_mins}")
        print(f"   Maxs: {self.band_maxs}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Cargar tile del Zarr
        tile = torch.from_numpy(self.tiles[idx][:12].astype("float32"))  # (12, 64, 64)

        # Normalizar a [0, 1]
        mins = torch.from_numpy(self.band_mins).float().view(-1, 1, 1)
        maxs = torch.from_numpy(self.band_maxs).float().view(-1, 1, 1)
        tile = (tile - mins) / (maxs - mins + 1e-8)
        tile = torch.clamp(tile, 0.0, 1.0)

        return {
            "tile": tile,
            "descripcion": row["descripcion"],
            "clase": row["clase"],
            "clase_idx": self.clase_a_idx[row["clase"]],
            "tile_id": row["tile_id"],
        }

### Crear dataloaders con WeightedRandomSampler (anti-desbalance)

In [29]:
# Cargar datasets
ds_train = GeoVisionDataset("train")
ds_val = GeoVisionDataset("val")
ds_test = GeoVisionDataset("test")

# WeightedRandomSampler para train (compensa desbalance 9.4:1 de suelo_urbano)
class_counts = ds_train.df["clase"].value_counts()
class_weights = {c: 1.0 / count for c, count in class_counts.items()}
sample_weights = ds_train.df["clase"].map(class_weights).values

train_sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights),
    num_samples=len(ds_train) * 2,  # oversampling 2x
    replacement=True,
)

# DataLoaders
train_loader = DataLoader(
    ds_train, batch_size=BATCH_SIZE, sampler=train_sampler,
    num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    ds_val, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)
test_loader = DataLoader(
    ds_test, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)

print(f"\n Dataloaders:")
print(f"   Train batches: {len(train_loader)} (oversampled 2x)")
print(f"   Val batches:   {len(val_loader)}")
print(f"   Test batches:  {len(test_loader)}")

FileNotFoundError: geovision-cali/datasets/sit2/metadatos.parquet

## 3. Construcción del Modelo

### 3.1 Encoder Visual: ViT-B/32 + BandAdapter

In [ ]:
print(" Construyendo encoder visual...")

# Cargar ViT-B/32 pre-entrenado (OpenCLIP como base; reemplazar por RemoteCLIP
# cuando los pesos estén disponibles)
vit_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
visual_backbone = vit_model.visual  # ViT puro, sin projection head
visual_backbone.output_dim = 512

# Congelar backbone
for p in visual_backbone.parameters():
    p.requires_grad = False

# BandAdapter: 12 bandas S2 → 3 canales RGB-equivalentes
band_adapter = BandAdapter(in_bands=12, out_channels=3)

# Validar dimensión de salida
dummy = torch.randn(2, 12, 64, 64)
with torch.no_grad():
    rgb = band_adapter(dummy)
    print(f"   BandAdapter: {dummy.shape} → {rgb.shape}")

### 3.2 Encoder Textual: paraphrase-multilingual-MiniLM

In [ ]:
print(" Construyendo encoder textual...")

text_backbone = AutoModel.from_pretrained(TEXT_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

# Proyectar de 384 → 512 para igualar dimensión con el visual
class TextEncoderWithProj(nn.Module):
    def __init__(self, backbone, input_dim=384, output_dim=512):
        super().__init__()
        self.backbone = backbone
        self.proj = nn.Linear(input_dim, output_dim)

    def forward(self, **kwargs):
        out = self.backbone(**kwargs)
        h = out.last_hidden_state.mean(dim=1)  # mean pooling (B, 384)
        return self.proj(h)  # (B, 512)

text_encoder = TextEncoderWithProj(text_backbone, input_dim=TEXT_DIM, output_dim=512)
print(f"   Text encoder: {TEXT_MODEL_NAME}")
print(f"   Output dim:   {TEXT_DIM} → 512 (proyección)")

### 3.3 Ensamblar GeoVision-CLIP completo

In [ ]:
print("🔧 Ensamblando GeoVision-CLIP...")

class VisualEncoderWithAdapter(nn.Module):
    """Encoder visual: BandAdapter → ViT-B/32."""
    def __init__(self, adapter, backbone):
        super().__init__()
        self.adapter = adapter
        self.backbone = backbone

    def forward(self, x):
        # x: (B, 12, 64, 64)
        rgb = self.adapter(x)                      # (B, 3, 64, 64)
        rgb_resized = F.interpolate(rgb, size=224, mode='bilinear', align_corners=False)
        # ViT espera RGB normalizado; nuestras tiles ya están en [0,1]
        # Ajustar a formato CLIP: (B, 3, 224, 224) → normalizado
        rgb_normalized = (rgb_resized - 0.5) / 0.5  # → [-1, 1] aprox
        return self.backbone(rgb_normalized)         # (B, 512)

visual_encoder = VisualEncoderWithAdapter(band_adapter, visual_backbone)

model = GeoVisionCLIP(
    visual_encoder=visual_encoder,
    text_encoder=text_encoder,
    text_tokenizer=tokenizer,
    visual_dim=VISUAL_DIM,
    text_dim=512,      # después de proyección interna
    sae_dim=SAE_DIM,
    proj_dim=PROJ_DIM,
    lambda_l1=LAMBDA_L1,
).to(DEVICE)

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Total parámetros:    {total_params:,}")
print(f"   Entrenables:          {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")

### 3.4 Loss y Optimizer

In [ ]:
criterion = GeoVisionCLIPLoss(temperature=TEMPERATURE, alpha=ALPHA)
criterion.to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)

# Cosine annealing con warmup
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

warmup_scheduler = LinearLR(
    optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS * len(train_loader)
)
cosine_scheduler = CosineAnnealingLR(
    optimizer, T_max=(EPOCHS - WARMUP_EPOCHS) * len(train_loader)
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP_EPOCHS * len(train_loader)],
)

scaler = GradScaler() if DEVICE.type == "cuda" else None

## 4. Entrenamiento

In [ ]:
print(" Iniciando entrenamiento...")
print(f"   Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR} | α: {ALPHA} | λ: {LAMBDA_L1}")
print(f"   Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"   Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")

# Tracking
history = {
    "train_loss": [], "train_clip": [], "train_sae_img": [], "train_sae_txt": [],
    "val_loss": [], "val_recall1": [], "val_recall5": [],
    "val_sparsity_img": [], "val_mse_rec_img": [],
    "lr": [],
}
best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0
t_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    # ─── TRAIN ───
    model.train()
    train_metrics = {"loss": 0.0, "clip": 0.0, "sae_img": 0.0, "sae_txt": 0.0}
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:2d}/{EPOCHS} [train]")
    for step, batch in enumerate(pbar):
        tiles = batch["tile"].to(DEVICE)
        texts = batch["descripcion"]

        with autocast(enabled=(scaler is not None)):
            outputs = model(tiles, texts)

            # Pérdidas SAE
            l_sae_img, l_rec_img, l_sparse_img = model.sae_img.loss(
                outputs["h_img"], outputs["z_img"], outputs["x_hat_img"]
            )
            l_sae_txt, l_rec_txt, l_sparse_txt = model.sae_txt.loss(
                outputs["h_txt"], outputs["z_txt"], outputs["x_hat_txt"]
            )

            # Pérdida total
            total, l_clip, _, _ = criterion(
                outputs["e_img"], outputs["e_txt"], l_sae_img, l_sae_txt
            )
            total = total / GRADIENT_ACCUMULATION_STEPS

        if scaler is not None:
            scaler.scale(total).backward()
        else:
            total.backward()

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

            # Normalizar decoders SAE (Zaigrajew et al., 2025)
            model.normalize_decoders()

        # Acumular métricas
        train_metrics["loss"] += total.item() * GRADIENT_ACCUMULATION_STEPS
        train_metrics["clip"] += l_clip.item()
        train_metrics["sae_img"] += l_sae_img.item()
        train_metrics["sae_txt"] += l_sae_txt.item()

        pbar.set_postfix(
            loss=f"{total.item()*GRADIENT_ACCUMULATION_STEPS:.3f}",
            clip=f"{l_clip.item():.3f}",
            sae_i=f"{l_sae_img.item():.3f}",
        )

    n_batches = len(train_loader)
    for k in train_metrics:
        train_metrics[k] /= n_batches
        history[f"train_{k}"].append(train_metrics[k])
    history["lr"].append(scheduler.get_last_lr()[0])

    # ─── VALIDATION ───
    model.eval()
    val_metrics = {"loss": 0.0}
    all_kpis = {"recall@1": [], "recall@5": [], "sparsity_img": [], "mse_rec_img": []}

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch:2d}/{EPOCHS} [val]  ", leave=False):
            tiles = batch["tile"].to(DEVICE)
            texts = batch["descripcion"]

            outputs = model(tiles, texts)

            l_sae_img, _, _ = model.sae_img.loss(
                outputs["h_img"], outputs["z_img"], outputs["x_hat_img"]
            )
            l_sae_txt, _, _ = model.sae_txt.loss(
                outputs["h_txt"], outputs["z_txt"], outputs["x_hat_txt"]
            )
            total, l_clip, _, _ = criterion(
                outputs["e_img"], outputs["e_txt"], l_sae_img, l_sae_txt
            )

            val_metrics["loss"] += total.item()

            # KPIs
            kpis = compute_all_kpis(
                outputs["e_img"], outputs["e_txt"],
                outputs["z_img"], outputs["z_txt"],
                outputs["h_img"], outputs["h_txt"],
                outputs["x_hat_img"], outputs["x_hat_txt"],
            )
            for k, v in kpis.items():
                if k in all_kpis:
                    all_kpis[k].append(v)

    val_metrics["loss"] /= len(val_loader)
    val_recall1 = np.mean(all_kpis["recall@1"])
    val_recall5 = np.mean(all_kpis["recall@5"])
    val_sparsity = np.mean(all_kpis["sparsity_img"])
    val_mse = np.mean(all_kpis["mse_rec_img"])

    history["val_loss"].append(val_metrics["loss"])
    history["val_recall1"].append(val_recall1)
    history["val_recall5"].append(val_recall5)
    history["val_sparsity_img"].append(val_sparsity)
    history["val_mse_rec_img"].append(val_mse)

    # ─── LOG ───
    dt = time.perf_counter() - t_start
    print(
        f"Epoch {epoch:2d}/{EPOCHS} | "
        f"Loss: {train_metrics['loss']:.4f} → {val_metrics['loss']:.4f} | "
        f"R@1: {val_recall1:.3f} | R@5: {val_recall5:.3f} | "
        f"Spars: {val_sparsity:.3f} | MSE_rec: {val_mse:.4f} | "
        f"{dt:.0f}s"
    )

    # ─── CHECKPOINT ───
    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_epoch = epoch
        patience_counter = 0

        ckpt_path = OUTPUT_DIR / "geovision_clip_best.pt"
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_metrics["loss"],
                "history": history,
                "seed": SEED,
                "config": {
                    "visual_dim": VISUAL_DIM,
                    "sae_dim": SAE_DIM,
                    "proj_dim": PROJ_DIM,
                    "lambda_l1": LAMBDA_L1,
                    "alpha": ALPHA,
                },
            },
            ckpt_path,
        )

        # MD5 del checkpoint
        with open(ckpt_path, "rb") as f:
            md5 = hashlib.md5(f.read()).hexdigest()
        print(f"    Best checkpoint saved: {ckpt_path} (MD5: {md5})")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"  Early stopping at epoch {epoch} (patience={EARLY_STOPPING_PATIENCE})")
            break

print(f"\n Entrenamiento completado en {time.perf_counter() - t_start:.0f}s")
print(f"   Mejor epoch: {best_epoch} (val_loss={best_val_loss:.4f})")

## 5. Curvas de Entrenamiento

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

epochs_range = range(1, len(history["train_loss"]) + 1)

# Loss total
axes[0, 0].plot(epochs_range, history["train_loss"], label="Train", color="blue")
axes[0, 0].plot(epochs_range, history["val_loss"], label="Val", color="orange")
axes[0, 0].set_title("Loss Total (L_clip + α·L_sae)")
axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# InfoNCE
axes[0, 1].plot(epochs_range, history["train_clip"], color="blue")
axes[0, 1].set_title("Loss InfoNCE (Train)")
axes[0, 1].set_xlabel("Epoch"); axes[0, 1].grid(alpha=0.3)

# SAE losses
axes[0, 2].plot(epochs_range, history["train_sae_img"], label="SAE Img", color="green")
axes[0, 2].plot(epochs_range, history["train_sae_txt"], label="SAE Txt", color="red")
axes[0, 2].set_title("Loss SAE (Train)")
axes[0, 2].set_xlabel("Epoch"); axes[0, 2].legend(); axes[0, 2].grid(alpha=0.3)

# Recall@1 y @5
axes[1, 0].plot(epochs_range, history["val_recall1"], label="R@1", color="blue", marker="o", markersize=3)
axes[1, 0].plot(epochs_range, history["val_recall5"], label="R@5", color="green", marker="o", markersize=3)
axes[1, 0].axhline(y=0.45, color="red", linestyle="--", alpha=0.5, label="R@1 mín")
axes[1, 0].axhline(y=0.70, color="orange", linestyle="--", alpha=0.5, label="R@5 mín")
axes[1, 0].set_title("Recall@k (Val) — KPIs")
axes[1, 0].set_xlabel("Epoch"); axes[1, 0].set_ylabel("Recall")
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

# Sparsity
axes[1, 1].plot(epochs_range, history["val_sparsity_img"], color="purple", marker="o", markersize=3)
axes[1, 1].axhline(y=0.70, color="red", linestyle="--", alpha=0.5, label="Mín (0.70)")
axes[1, 1].axhline(y=0.85, color="green", linestyle="--", alpha=0.5, label="Exc (0.85)")
axes[1, 1].set_title("Sparsity Ratio SAE Visual (Val) — KPI")
axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_ylabel("Sparsity")
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

# MSE reconstrucción
axes[1, 2].plot(epochs_range, history["val_mse_rec_img"], color="brown", marker="o", markersize=3)
axes[1, 2].axhline(y=0.05, color="red", linestyle="--", alpha=0.5, label="Máx (0.05)")
axes[1, 2].axhline(y=0.02, color="green", linestyle="--", alpha=0.5, label="Exc (0.02)")
axes[1, 2].set_title("MSE Reconstrucción SAE (Val) — KPI")
axes[1, 2].set_xlabel("Epoch"); axes[1, 2].set_ylabel("MSE")
axes[1, 2].legend(); axes[1, 2].grid(alpha=0.3)

plt.suptitle("GeoVision-CLIP Cali — Curvas de Entrenamiento (Situación 2)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Evaluación Final en Test

In [ ]:
print(" Evaluación en test set...")

model.eval()
test_outputs = {"e_img": [], "e_txt": [], "z_img": [], "z_txt": [],
                "h_img": [], "h_txt": [], "x_hat_img": [], "x_hat_txt": [],
                "clases": [], "tile_ids": []}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test"):
        tiles = batch["tile"].to(DEVICE)
        texts = batch["descripcion"]

        outputs = model(tiles, texts)

        for k in ["e_img", "e_txt", "z_img", "z_txt", "h_img", "h_txt", "x_hat_img", "x_hat_txt"]:
            test_outputs[k].append(outputs[k].cpu())
        test_outputs["clases"].extend(batch["clase"])
        test_outputs["tile_ids"].extend(batch["tile_id"])

# Concatenar
for k in ["e_img", "e_txt", "z_img", "z_txt", "h_img", "h_txt", "x_hat_img", "x_hat_txt"]:
    test_outputs[k] = torch.cat(test_outputs[k], dim=0)

# Calcular KPIs finales
test_kpis = compute_all_kpis(
    test_outputs["e_img"], test_outputs["e_txt"],
    test_outputs["z_img"], test_outputs["z_txt"],
    test_outputs["h_img"], test_outputs["h_txt"],
    test_outputs["x_hat_img"], test_outputs["x_hat_txt"],
)

print("\n KPIs Finales (Test Set):")
print("═" * 45)
kpi_umbrales = {
    "recall@1": (0.45, 0.65),
    "recall@5": (0.70, 0.85),
    "sparsity_img": (0.70, 0.85),
    "sparsity_txt": (0.70, 0.85),
    "mse_rec_img": (0.05, 0.02),
    "mse_rec_txt": (0.05, 0.02),
}
for kpi, (minimo, excelente) in kpi_umbrales.items():
    val = test_kpis.get(kpi, float("nan"))
    if kpi.startswith("mse"):
        status = " EXCELENTE" if val <= excelente else (" MÍNIMO" if val <= minimo else " NO ALCANZA")
    else:
        status = " EXCELENTE" if val >= excelente else (" MÍNIMO" if val >= minimo else " NO ALCANZA")
    print(f"   {kpi:20s} = {val:.4f}  [{status}]")

print(f"\n   cos_sim_rec_img     = {test_kpis.get('cos_sim_rec_img', 0):.4f}")
print(f"   fvu_img             = {test_kpis.get('fvu_img', 0):.4f}")

# Per-class Recall@1
print("\n Recall@1 por Clase:")
sim = test_outputs["e_img"] @ test_outputs["e_txt"].T
for i, clase in enumerate(CLASES):
    mask = np.array([c == clase for c in test_outputs["clases"]])
    if mask.sum() > 0:
        clase_sim = sim[mask]
        r1 = (clase_sim.argmax(dim=1) == torch.arange(clase_sim.size(0))).float().mean().item()
        print(f"   {clase:30s} = {r1:.3f}  (n={mask.sum()})")

## 7. Guardar Checkpoint Final + MD5

In [ ]:
final_ckpt = OUTPUT_DIR / "geovision_clip_final.pt"
torch.save(
    {
        "epoch": best_epoch,
        "model_state_dict": model.state_dict(),
        "test_kpis": test_kpis,
        "history": history,
        "seed": SEED,
        "timestamp": datetime.now().isoformat(),
    },
    final_ckpt,
)

with open(final_ckpt, "rb") as f:
    md5_final = hashlib.md5(f.read()).hexdigest()

# Guardar MD5 en archivo separado para verificación entre integrantes
(OUTPUT_DIR / "checkpoint_md5.txt").write_text(
    f"geovision_clip_final.pt  MD5: {md5_final}\n"
    f"Timestamp: {datetime.now().isoformat()}\n"
    f"Test Recall@1: {test_kpis['recall@1']:.4f}\n"
    f"Test Recall@5: {test_kpis['recall@5']:.4f}\n"
    f"Sparsity: {test_kpis['sparsity_img']:.4f}\n"
    f"MSE rec: {test_kpis['mse_rec_img']:.4f}\n"
)

# Guardar KPIs como JSON
(OUTPUT_DIR / "test_kpis.json").write_text(
    json.dumps(test_kpis, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f" Checkpoint final: {final_ckpt}")
print(f"   MD5: {md5_final}")
print(f"   Archivos guardados en {OUTPUT_DIR}/:")
for f in sorted(OUTPUT_DIR.glob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"   - {f.name} ({size_mb:.1f} MB)")

print("\n Entrenamiento Situación 2 completado!")